# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² (Frontiers) mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object is a Dataclass, access its attributes directly (not by subscript)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We iterate over all record sets in the dataset and print their `@id`, name, and included fields. Each field is also identified by its `@id` (as per Croissant best practices).

In [ ]:
# List all record sets and their details
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets (tables):\n")
for rset in record_sets:
    print(f'- RecordSet @id: {rset.id}')
    print(f'  Name: {rset.name}')
    print(f'  Description: {rset.description}')
    print('  Fields:')
    for fld in rset.fields:
        print(f'    - Field @id: {fld.id}, name: {fld.name}, dataType: {fld.data_type}')
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all record sets into pandas DataFrames. Record sets and fields are referenced by their Croissant `@id` values.

In [ ]:
# Create a list of record set @ids
record_set_ids = [rset.id for rset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record_set_id is used as input for dataset.records
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record set {record_set_id}")

# Pick the first record set for display
first_record_set_id = record_set_ids[0]
print(f"Columns in record set {first_record_set_id}:")
print(dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a representative numeric field for demonstration. We'll use its field `@id` and show filtering, normalization, and grouping by another field.

Replace `your_numeric_field_id` and `your_grouping_field_id` with corresponding `@id`s from the printed overview above (for demonstration, we'll auto-pick the first matching field).

In [ ]:
# Choose a numeric field for filtering/normalizing
# Here we programmatically select the first Float or Integer field
first_rset = record_sets[0]
numeric_field_id = None
for f in first_rset.fields:
    if f.data_type in ('Float', 'Integer', 'Number'):
        numeric_field_id = f.id
        break
if numeric_field_id is None:
    raise Exception('No numeric field found for EDA!')
print(f"Chosen numeric field: {numeric_field_id}")

df = dataframes[first_record_set_id]

# Drop NAs for the field
numeric_ser = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = numeric_ser.mean()  # Use mean as threshold
filtered_df = df[numeric_ser > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize and add a new column
filtered_df[f"{numeric_field_id}_normalized"] = (numeric_ser[numeric_ser > threshold] - numeric_ser.mean()) / numeric_ser.std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Select group field (categorical/text)
group_field_id = None
for f in first_rset.fields:
    if f.data_type in ('Text', 'String') and f.id != numeric_field_id:
        group_field_id = f.id
        break
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} in '{first_record_set_id}'")
plt.tight_layout()
plt.show()

# If there is a grouping field, plot boxplot
if group_field_id is not None and group_field_id in df.columns:
    # Use only first 10 groups for clarity
    top_groups = df[group_field_id].value_counts().index[:10]
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
    plt.title(f"{numeric_field_id} by {group_field_id} (first 10)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've used `mlcroissant` to load the FAIR² mass spectrometry dataset, explored its structure by referencing all fields by their Croissant `@id`, extracted records for analysis, processed and visualized protein-level data. This approach can be generalized to reproducibly access, process, and analyze cross-domain datasets adhering to the Croissant standard.